**Last modified on**: 16/04/2026

**Author**: Onur Serçinoğlu

**Update Logs**:

16/04/2026: Initial version

**Credits**:

This Jupyter notebook accompanies the HMMER & Profile Hidden Markov Models lecture prepared for BENG451/BSB511 at Gebze Technical University. Key references:
- Eddy SR (1998). Profile hidden Markov models. Bioinformatics 14:755–763.
- Eddy SR (2011). Accelerated profile HMM searches. PLoS Comput Biol 7:e1002195.
- Mistry J et al. (2021). Pfam: The protein families database in 2021. Nucleic Acids Res 49:D412–D419.
- Durbin R et al. (1998). Biological sequence analysis. Cambridge University Press.

# Building a Profile HMM with hmmbuild

In the previous notebook (`6_1_hmm_theory.ipynb`) we studied the mathematical foundations of profile HMMs — Match/Insert/Delete states, emission probabilities, transition probabilities, Viterbi, and Forward algorithms.

In this notebook we move from theory to practice. We will:

1. Load and inspect the globin multiple sequence alignment we will use as input.
2. Run **`hmmbuild`** to estimate the profile HMM parameters from that alignment.
3. Explore the resulting `.hmm` text file — emission probabilities and transition probabilities.
4. Run **`hmmpress`** to prepare the model for fast database searching.
5. Validate the model with a self-search (our alignment sequences searched against the model we just built).

The next notebook (`6_3_hmmsearch.ipynb`) will use this model to search a protein database.

In [ ]:
import os

# ─── Adjustable paths ────────────────────────────────────────────────────────
# Edit PFAM_HMM_PATH to point to your local Pfam-A.hmm if running outside
# the default course environment.
PFAM_HMM_PATH = "/mnt/c/Users/onurs/Downloads/Pfam-A.hmm"
WORK_DIR = "hmmer_build_work"
ALIGNMENT_FILE = "globin_alignment_pam250.fa"   # already in exercises/
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(WORK_DIR, exist_ok=True)
print(f"Working directory: {os.path.abspath(WORK_DIR)}")
print(f"Input alignment:   {os.path.abspath(ALIGNMENT_FILE)}")

---

## Section 1: Load and Inspect the Alignment

### The globin family

Globins are a superfamily of oxygen-binding proteins found in virtually all kingdoms of life. The hallmark of the fold is the **globin fold**: eight alpha-helices (conventionally labelled A–H) that form a hydrophobic pocket housing the heme group. Despite enormous sequence divergence — human hemoglobin and leghemoglobin (from plant root nodules) share less than 20% sequence identity — the three-dimensional structure is highly conserved.

The alignment we are using (`globin_alignment_pam250.fa`) contains two sequences:
- **NP_000509.1** — human hemoglobin subunit beta (HbB, *Homo sapiens*)
- **NP_005359.1** — human myoglobin isoform 1 (Mb, *Homo sapiens*)

Although this is a small alignment (two sequences), it is sufficient to illustrate the hmmbuild workflow. In practice, globin family models in Pfam contain hundreds of sequences.

**Key idea:** `hmmbuild` reads the alignment and asks, for each column: "What fraction of sequences have a residue here (vs. a gap)?" Columns where most sequences have a residue become **match states** (consensus positions); columns dominated by gaps become **insert states**.

In [ ]:
import subprocess
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import AlignIO, SeqIO

# Load the globin alignment using Bio.AlignIO
# The FASTA format requires AlignIO to know the format explicitly.
alignment = AlignIO.read(ALIGNMENT_FILE, "fasta")

num_sequences = len(alignment)
alignment_length = alignment.get_alignment_length()

print("Globin alignment summary")
print("=" * 50)
print(f"Number of sequences : {num_sequences}")
print(f"Alignment length    : {alignment_length} columns")
print()
print("Sequence IDs:")
for record in alignment:
    print(f"  {record.id:30s}  length={len(record.seq)}  (ungapped={len(str(record.seq).replace('-', ''))})") 

In [ ]:
# Compute per-column gap fraction using numpy
# For each alignment column, count how many sequences have a gap character ('-')

alignment_array = np.array([list(str(record.seq)) for record in alignment])
# Shape: (num_sequences, alignment_length)

gap_counts_per_column = np.sum(alignment_array == '-', axis=0)   # sum along sequences
gap_fraction_per_column = gap_counts_per_column / num_sequences

print(f"Alignment array shape   : {alignment_array.shape}  (sequences x columns)")
print(f"Number of columns       : {alignment_length}")
print(f"Mean gap fraction       : {gap_fraction_per_column.mean():.3f}")
print(f"Fully gapped columns    : {int(np.sum(gap_fraction_per_column == 1.0))}")
print(f"Columns with gap >= 0.5 : {int(np.sum(gap_fraction_per_column >= 0.5))}  (will become insert states)")
print(f"Columns with gap <  0.5 : {int(np.sum(gap_fraction_per_column < 0.5))}  (will become match states)")

In [ ]:
# Plot gap fraction per alignment column
# HMMER's default rule: columns with gap fraction >= 0.5 are NOT consensus match columns.

column_indices = np.arange(alignment_length)

fig, ax = plt.subplots(figsize=(14, 4))

# Color bars: below threshold = match state (teal), at or above = insert state (salmon)
colors = ['#2a7f8a' if gf < 0.5 else '#e07070' for gf in gap_fraction_per_column]
ax.bar(column_indices, gap_fraction_per_column, color=colors, width=1.0, edgecolor='none')

# Horizontal threshold line
ax.axhline(y=0.5, color='black', linestyle='--', linewidth=1.2,
           label='0.5 threshold (HMMER default)')

ax.set_title(
    "Gap fraction per alignment column — columns above 0.5 become insert states",
    fontsize=12
)
ax.set_xlabel("Alignment column (0-indexed)", fontsize=11)
ax.set_ylabel("Gap fraction (0 = no gaps, 1 = all gaps)", fontsize=11)
ax.set_xlim(-1, alignment_length)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)

# Custom legend patches for column types
from matplotlib.patches import Patch
legend_patches = [
    Patch(facecolor='#2a7f8a', label='Match state (gap < 0.5)'),
    Patch(facecolor='#e07070', label='Insert state (gap >= 0.5)'),
    Patch(facecolor='none', edgecolor='black', linestyle='--', label='0.5 threshold'),
]
ax.legend(handles=legend_patches, fontsize=10, loc='upper right')

plt.tight_layout()
plt.show()

**Question:** Looking at the gap fraction plot, which regions of the alignment are highly gapped? What might this tell us about the protein structure?

*Hint:* Loops and disordered regions connecting secondary structure elements tend to be poorly conserved in length — one protein may have an insertion that the other lacks. Highly gapped columns often correspond to surface-exposed loops. Conserved (low-gap) columns tend to correspond to elements important for the fold, such as buried alpha-helices or the heme-binding pocket.

---

## Section 2: Run hmmbuild

### What does hmmbuild do?

`hmmbuild` reads a multiple sequence alignment and constructs a **profile HMM** from it. The key steps it performs are:

1. **Column classification**: By default, columns where fewer than 50% of sequences have a gap are designated as **match columns** (consensus positions). The rest become insert columns.

2. **Probability estimation**: For each match state, it counts how often each of the 20 amino acids occurs in that column across all sequences. These raw counts are then regularised using **Dirichlet mixture priors** — a Bayesian technique that prevents probabilities from being zero when an amino acid is unseen in the alignment (the so-called zero-count problem). The result is a position-specific emission probability vector of length 20.

3. **Transition probability estimation**: `hmmbuild` also estimates the M→M, M→I, M→D, I→M, I→I, D→M, D→D transition probabilities at each node from the observed pattern of gaps in the alignment.

4. **Effective sequence number (neff)**: If the alignment sequences are highly redundant (e.g., 90% identical), `hmmbuild` downweights them so that redundant sequences do not dominate the probability estimates. The `neff` value shown in the output is the effective number of sequences after this weighting — it is often less than the actual count.

5. **E-value calibration**: `hmmbuild` performs a brief forward/Viterbi simulation to calibrate the statistical parameters needed for E-value calculation during `hmmsearch`.

In [ ]:
# Run hmmbuild to build the profile HMM from the globin alignment
hmm_file = os.path.join(WORK_DIR, "globin.hmm")

result = subprocess.run(
    ['hmmbuild', hmm_file, ALIGNMENT_FILE],
    capture_output=True,
    text=True
)

print("hmmbuild stdout:")
print(result.stdout)

if result.returncode != 0:
    print("ERROR: hmmbuild exited with return code", result.returncode)
    print(result.stderr)
else:
    print(f"hmmbuild completed successfully. Output file: {hmm_file}")
    print(f"File size: {os.path.getsize(hmm_file)} bytes")

In [ ]:
# Parse key statistics from the hmmbuild stdout
# We are looking for lines like:
#   Number of sequences:     2
#   Effective number of seqs:  1.40  (hmmbuild calls this 'neff')
#   Model length:           149

stdout_text = result.stdout

# Extract number of sequences
match_nseq = re.search(r'Number of sequences:\s+(\d+)', stdout_text)
num_seqs_hmmbuild = int(match_nseq.group(1)) if match_nseq else None

# Extract effective number of sequences (neff)
# In newer HMMER versions the label may vary slightly — try a few patterns
match_neff = re.search(r'Effective (?:number of )?(?:seq(?:uence)?s?|seqs)[^:]*:\s+([\d.]+)', stdout_text, re.IGNORECASE)
neff = float(match_neff.group(1)) if match_neff else None

# Extract model length
match_mlen = re.search(r'Model length:\s+(\d+)', stdout_text)
model_length = int(match_mlen.group(1)) if match_mlen else None

print("hmmbuild key statistics")
print("=" * 45)
print(f"  Number of input sequences : {num_seqs_hmmbuild}")
print(f"  Effective number (neff)   : {neff}")
print(f"  Model length (# of match states) : {model_length}")

if neff is not None and num_seqs_hmmbuild is not None:
    ratio = neff / num_seqs_hmmbuild
    print()
    print(f"  neff / actual count = {ratio:.3f}")
    if ratio < 1.0:
        print("  => Sequences are downweighted due to redundancy.")
    else:
        print("  => No downweighting applied (neff >= actual count).")

**Question:** The 'effective number of sequences' is often less than the actual number. Why might HMMER downweight redundant sequences? (Hint: think about what happens if all your alignment sequences are nearly identical.)

*Answer hint:* If you have 100 nearly identical sequences, including all of them at full weight would make the model look artificially confident — even rare amino acids that appear in only those 100 (but no independent lineages) would receive unfairly high probability. Downweighting treats groups of highly similar sequences as contributing only a fraction of a count each, so the model reflects true independent evolutionary observations rather than sampling bias.

---

## Section 3: Explore the .hmm File

### The HMMER3 file format

The `.hmm` file produced by `hmmbuild` is a **human-readable text file** containing all the parameters of the model. It is divided into two sections:

**Header section** (before the `HMM` keyword line):
- `NAME`, `ACC`, `DESC` — model name, accession, and description
- `LENG` — model length (number of match states)
- `ALPH` — alphabet (amino or DNA)
- `DATE`, `NSEQ`, `EFFN` — build date, number of sequences, effective number
- `CKSUM` — alignment checksum
- Statistical calibration parameters (`MSV`, `VITERBI`, `FORWARD`)

**Body section** (after the `HMM` keyword line and the two transition-header lines):
- First comes a `COMPO` line — the model's overall amino acid composition
- Then one **3-line block per node** (node 1 through node L):
  1. **Match emission line**: node number, then 20 values for A C D E F G H I K L M N P Q R S T V W Y
  2. **Insert emission line**: 20 values (same alphabet)
  3. **Transition probability line**: 7 values for m→m m→i m→d i→m i→i d→m d→d

All probability values in the body are stored as **negative log-probabilities scaled by an internal constant** (integers). To recover the actual probability from a stored value `v`, the approximate conversion is:

```
probability ≈ exp(-v / 1000.0)
```

The exact scale and encoding are documented in the HMMER User Guide, Appendix B.

In [ ]:
# Open the .hmm file and print the header section
# We print lines up to and including the first 'HMM' keyword line (approx. first 25 lines)

with open(hmm_file) as f:
    all_hmm_lines = f.readlines()

print(f"Total lines in {hmm_file}: {len(all_hmm_lines)}")
print()
print("Header section of the .hmm file:")
print("-" * 70)

# Print until we hit the HMM keyword line (and include it), up to a maximum of 35 lines
header_end_idx = 0
for idx, line in enumerate(all_hmm_lines):
    print(line, end='')
    if line.startswith('HMM '):
        header_end_idx = idx
        break
    if idx >= 34:   # safety cap
        header_end_idx = idx
        break

print("-" * 70)
print(f"(Header ends at line {header_end_idx}; body starts after the transition-labels line.)")

In [ ]:
# Parse emission probability lines for ALL match states
#
# HMMER .hmm body structure after the HMM keyword section:
#   COMPO  <20 values>       <- composite background composition
#          <20 values>       <- insert emissions for node 0 (before node 1)
#          <7 values>        <- transitions from node 0
#   1      <20 values>       <- match emissions for node 1   <-- we want these
#          <20 values>       <- insert emissions for node 1
#          <7 values>        <- transitions for node 1
#   2      <20 values>       <- match emissions for node 2
#   ...
#
# A match emission line starts with an INTEGER (the node number).

amino_acids = list("ACDEFGHIKLMNPQRSTVWY")  # HMMER's alphabet order

match_emissions = {}   # node_num (int) -> list of 20 probabilities

# Find where the HMM body starts: look for the line that says 'HMM'
body_start_idx = None
for idx, line in enumerate(all_hmm_lines):
    if line.strip().startswith('HMM '):
        # Skip the 'HMM' line itself and the next header line (amino acid labels)
        body_start_idx = idx + 2
        break

if body_start_idx is None:
    print("ERROR: Could not find HMM keyword in the file.")
else:
    print(f"HMM body starts at line index {body_start_idx}.")
    print()

    # Iterate through body lines and capture match emission lines
    # A match emission line's first token is a decimal integer (node number)
    for line in all_hmm_lines[body_start_idx:]:
        stripped = line.strip()
        if stripped == '' or stripped == '//':
            continue
        tokens = stripped.split()
        # Match emission line: first token is a plain integer node number
        if tokens[0].isdigit():
            node_num = int(tokens[0])
            # Next 20 tokens are the emission values
            # Some may be '*' meaning -infinity (zero probability) -- treat as 0.0
            probs = []
            for val in tokens[1:21]:
                if val == '*':
                    probs.append(0.0)
                else:
                    # Convert from stored negative log-probability to probability:
                    # prob = exp(-float(val) / 1000.0)
                    probs.append(np.exp(-float(val) / 1000.0))
            if len(probs) == 20:
                match_emissions[node_num] = probs

    print(f"Parsed match emissions for {len(match_emissions)} nodes.")
    print()

    # Show top-3 amino acids for the first 5 nodes
    print("Top-3 amino acids for the first 5 match states:")
    print("-" * 55)
    for node_num in sorted(match_emissions.keys())[:5]:
        probs = match_emissions[node_num]
        aa_probs = sorted(zip(amino_acids, probs), key=lambda x: x[1], reverse=True)
        top3 = ', '.join(f"{aa}({p:.3f})" for aa, p in aa_probs[:3])
        print(f"  Node {node_num:4d}: {top3}")

In [ ]:
# Plot emission probability distributions for nodes 1, 50, and last node
# to compare conserved vs. variable positions

all_node_nums = sorted(match_emissions.keys())
last_node = all_node_nums[-1]

# Choose three representative positions
mid_node = 50 if 50 in match_emissions else all_node_nums[len(all_node_nums) // 2]
positions_to_plot = [1, mid_node, last_node]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(
    "Emission probability distributions at three match state positions",
    fontsize=12
)

colors = ['#2a7f8a', '#e07070', '#6a8f3a']
position_labels = {
    positions_to_plot[0]: f"Node {positions_to_plot[0]} (N-terminus)",
    positions_to_plot[1]: f"Node {positions_to_plot[1]} (middle)",
    positions_to_plot[2]: f"Node {positions_to_plot[2]} (C-terminus / last)",
}

for ax, node_num, color in zip(axes, positions_to_plot, colors):
    probs = match_emissions[node_num]
    ax.bar(amino_acids, probs, color=color, edgecolor='black', linewidth=0.4)
    ax.set_title(position_labels[node_num], fontsize=10)
    ax.set_xlabel("Amino acid", fontsize=9)
    ax.set_ylabel("Probability", fontsize=9)
    ax.tick_params(axis='x', labelsize=8)

    # Annotate the most probable amino acid
    top_aa = amino_acids[int(np.argmax(probs))]
    top_p = max(probs)
    ax.annotate(
        f"{top_aa} ({top_p:.3f})",
        xy=(amino_acids.index(top_aa), top_p),
        xytext=(amino_acids.index(top_aa) + 1.5, top_p * 0.9),
        fontsize=8,
        arrowprops=dict(arrowstyle='->', color='black', lw=0.8)
    )

plt.tight_layout()
plt.show()

# Print entropy (as a measure of conservation) for each position
print("Conservation measure (entropy) per position:")
print("  Lower entropy = more conserved")
print()
for node_num in positions_to_plot:
    probs = np.array(match_emissions[node_num])
    # Remove zeros before log to avoid nan
    nonzero = probs[probs > 0]
    entropy = -np.sum(nonzero * np.log2(nonzero))
    top_aa = amino_acids[int(np.argmax(probs))]
    print(f"  Node {node_num:4d}: entropy = {entropy:.3f} bits,  most probable aa = {top_aa} ({max(probs):.3f})")

**Note on the emission probability conversion:** HMMER3 stores match emission values as integers representing `-ln(p)` multiplied by an internal scale factor. The approximation `prob ≈ exp(-v / 1000.0)` is used here for illustration. For exact values and the precise encoding, see the HMMER User Guide, Appendix B ("HMMER3 file formats").

**Question:** Compare the emission probability distributions for the first position (N-terminus), a middle position, and the last position. Which position looks most conserved? Can you infer what amino acid is most critical there?

*Hint:* A conserved position has one amino acid with a notably higher probability than all others — the distribution is "peaked". A variable position has a flatter distribution. The conservation reflects biological constraint: positions buried in the hydrophobic core, or directly involved in heme binding (e.g., the proximal His) tend to be strongly conserved.

---

## Section 4: Parse Transition Probabilities

### Position-specific gap penalties

One of the most powerful features of profile HMMs compared to BLAST is **position-specific gap penalties**. In BLAST, the same gap-open and gap-extend penalty applies everywhere. In a profile HMM, the transition probabilities at each node encode how likely a gap is at that exact position in the family.

For each node `k`, the transition probability line contains 7 values:

| Index | Transition | Meaning |
|-------|-----------|--------|
| 0 | M→M | Probability of staying on the "consensus highway" |
| 1 | M→I | Probability of opening an insertion after this position |
| 2 | M→D | Probability of skipping (deleting) the next position |
| 3 | I→M | Probability of returning to match from an insertion |
| 4 | I→I | Probability of extending an insertion (staying in insert state) |
| 5 | D→M | Probability of returning to match from a deletion |
| 6 | D→D | Probability of extending a deletion |

A high M→M probability at a node means the family very rarely has a gap there — effectively a high gap penalty. A high M→I probability means insertions are common after that position (a structurally tolerant region).

In [ ]:
# Parse transition probability lines for each node
#
# In the .hmm body, each node block has 3 lines:
#   Line 1: match emission    (starts with node integer)
#   Line 2: insert emission   (starts with whitespace)
#   Line 3: transition probs  (starts with whitespace)
#
# We iterate through the body in groups of 3 lines (skipping COMPO block)

transition_col_names = ['M->M', 'M->I', 'M->D', 'I->M', 'I->I', 'D->M', 'D->D']

rows = []   # list of dicts for DataFrame

# We'll track the body line state: after COMPO (which has its own 3-line block),
# every 3 lines belongs to one node.

body_lines = all_hmm_lines[body_start_idx:]

# Filter out empty/comment lines and '//' terminator
body_lines_clean = [
    line for line in body_lines
    if line.strip() != '' and not line.strip().startswith('//')
]

# The body is grouped in 3-line blocks. Skip the first block (COMPO + 2 lines).
# Starting from block index 1, every block is a node.
num_blocks = len(body_lines_clean) // 3
print(f"Total 3-line blocks in body (including COMPO): {num_blocks}")
print(f"Node blocks: {num_blocks - 1}")
print()

for block_idx in range(1, num_blocks):   # skip block 0 (COMPO)
    line1 = body_lines_clean[block_idx * 3].strip().split()     # match
    # line2 = insert (not needed here)
    line3 = body_lines_clean[block_idx * 3 + 2].strip().split() # transitions

    node_num = int(line1[0])

    row = {'node': node_num}
    for col_idx, col_name in enumerate(transition_col_names):
        val = line3[col_idx]
        if val == '*':
            row[col_name] = 0.0
        else:
            row[col_name] = np.exp(-float(val) / 1000.0)
    rows.append(row)

transitions_df = pd.DataFrame(rows)

print("Transition probability DataFrame (first 10 rows):")
print(transitions_df.head(10).to_string(index=False))
print()
print(f"Total rows: {len(transitions_df)}")
print()
print("Summary statistics for M->M and M->I:")
print(transitions_df[['M->M', 'M->I']].describe().to_string())

In [ ]:
# Plot M->M and M->I transition probabilities across all nodes
# M->M = probability of staying on the consensus highway (high = conserved position)
# M->I = probability of opening an insertion (high = variable loop region)

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    transitions_df['node'],
    transitions_df['M->M'],
    color='#2a7f8a',
    linewidth=1.5,
    label='M→M (stay on consensus highway)'
)
ax.plot(
    transitions_df['node'],
    transitions_df['M->I'],
    color='#e07070',
    linewidth=1.5,
    linestyle='--',
    label='M→I (open insertion)'
)

ax.set_title(
    "Transition probabilities across model nodes — where does the model expect insertions?",
    fontsize=12
)
ax.set_xlabel("Node (match state position)", fontsize=11)
ax.set_ylabel("Transition probability", fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print the top 5 positions with highest M->I (most likely to have insertions)
top_insertion_positions = transitions_df.nlargest(5, 'M->I')[['node', 'M->M', 'M->I', 'M->D']]
print("Top 5 nodes with highest M->I (insertion-prone positions):")
print(top_insertion_positions.to_string(index=False))

**Question:** Are there positions where M→I probability is notably higher? What might those positions correspond to structurally?

*Hint:* In globins, inter-helix loops are the main regions that vary in length between different family members. The canonical globin fold connects eight helices (A–H) with loops of variable length. Positions at the N- and C-terminal ends of helices, or in surface-exposed loops, tend to tolerate insertions more readily than positions deep in the hydrophobic core. Compare the positions of high M→I probability with the known secondary structure of hemoglobin or myoglobin.

---

## Section 5: Run hmmpress

### Why hmmpress?

The `.hmm` text file produced by `hmmbuild` is human-readable but not optimised for fast searching. Before running `hmmsearch` against a large sequence database, HMMER requires the model to be **compressed into binary index files** using `hmmpress`.

`hmmpress` produces four binary files from `model.hmm`:

| Extension | Contents |
|-----------|----------|
| `.h3i` | Index — maps model names to file offsets |
| `.h3m` | The model data in compact binary format |
| `.h3f` | Pre-computed MSV filter parameters (for the fast pre-filter step) |
| `.h3p` | Profile data for the main forward/Viterbi passes |

HMMER's search pipeline runs a three-stage filter: (1) MSV (a fast ungapped filter), (2) Viterbi (gapped), and (3) full Forward. The binary files allow HMMER to load only the relevant profile data for each stage without reading the entire file. For a database like Pfam-A with tens of thousands of models, this provides enormous speed gains.

If you only have one model (as in this exercise), `hmmpress` is technically optional for `hmmsearch` — but it is required for `hmmscan`.

In [ ]:
# Run hmmpress on the globin HMM file
# hmmpress will overwrite existing binary files if they exist (-f flag suppresses the warning)
result_press = subprocess.run(
    ['hmmpress', '-f', hmm_file],
    capture_output=True,
    text=True
)

print("hmmpress stdout:")
print(result_press.stdout)

if result_press.returncode != 0:
    print("ERROR: hmmpress exited with return code", result_press.returncode)
    print(result_press.stderr)
else:
    print("hmmpress completed successfully.")

In [ ]:
# Verify that all four binary index files were created, and print their sizes

expected_extensions = ['.h3i', '.h3m', '.h3f', '.h3p']
extension_descriptions = {
    '.h3i': 'Index (model name -> file offset)',
    '.h3m': 'Model data (compact binary)',
    '.h3f': 'MSV filter parameters (fast pre-filter)',
    '.h3p': 'Profile data (Viterbi / Forward passes)',
}

print("hmmpress output files:")
print("-" * 70)
print(f"{'File':<40}  {'Size (bytes)':>12}  Description")
print("-" * 70)

all_present = True
for ext in expected_extensions:
    binary_file = hmm_file + ext
    if os.path.exists(binary_file):
        size_bytes = os.path.getsize(binary_file)
        desc = extension_descriptions.get(ext, '')
        print(f"{binary_file:<40}  {size_bytes:>12}  {desc}")
    else:
        print(f"{binary_file:<40}  {'MISSING':>12}")
        all_present = False

print("-" * 70)
if all_present:
    print("All four binary index files are present. Model is ready for hmmsearch/hmmscan.")
else:
    print("WARNING: One or more binary files are missing. Re-run hmmpress.")

---

## Section 6: Validate with a Self-Search

### Sanity check: search the alignment sequences against the model

The simplest validation of a newly built profile HMM is a **self-search**: run `hmmsearch` using our globin model as the query and the same globin alignment sequences as the database. Because we built the model *from* these sequences, every sequence should score very well — tiny E-values, high bitscore.

If any of the input sequences scored poorly, it would indicate a bug in the alignment, a corrupt HMM file, or some other issue.

This kind of self-search is also useful as a calibration step: you can see the score distribution of known true positives, which helps you set thresholds for real database searches.

In [ ]:
# Run hmmsearch: globin.hmm (query) vs. globin_alignment_pam250.fa (database)
# --tblout: machine-readable per-sequence hit table

tblout_file = os.path.join(WORK_DIR, "self_search.tbl")

result_search = subprocess.run(
    ['hmmsearch', '--tblout', tblout_file, hmm_file, ALIGNMENT_FILE],
    capture_output=True,
    text=True
)

print("hmmsearch stdout (first 2000 chars):")
print(result_search.stdout[:2000])

if result_search.returncode != 0:
    print("ERROR: hmmsearch exited with return code", result_search.returncode)
    print(result_search.stderr)

In [ ]:
# Parse the hmmsearch --tblout file into a pandas DataFrame
#
# The tblout format:
#   - Lines starting with '#' are comments/headers
#   - Data lines: whitespace-separated fields
#     field[0]  = target sequence name
#     field[1]  = target accession (or '-')
#     field[2]  = query name
#     field[3]  = query accession (or '-')
#     field[4]  = full-sequence E-value
#     field[5]  = full-sequence score (bits)
#     field[6]  = full-sequence bias
#     field[7]  = best-domain E-value
#     field[8]  = best-domain score (bits)
#     field[9]  = best-domain bias
#     ... (more domain counts and description)

rows = []

with open(tblout_file) as f:
    for line in f:
        if line.startswith('#') or line.strip() == '':
            continue
        fields = line.split()
        rows.append({
            'sequence':           fields[0],
            'target_accession':   fields[1],
            'query_name':         fields[2],
            'query_accession':    fields[3],
            'full_seq_evalue':    float(fields[4]),
            'full_seq_score':     float(fields[5]),
            'full_seq_bias':      float(fields[6]),
            'best_domain_evalue': float(fields[7]),
            'best_domain_score':  float(fields[8]),
        })

hits_df = pd.DataFrame(rows).sort_values('full_seq_evalue').reset_index(drop=True)

print("Self-search results (sorted by E-value):")
print("=" * 80)
print(hits_df.to_string(index=False))
print()
print(f"Total hits: {len(hits_df)}")

In [ ]:
# Check that all hits have E-value < 0.01 (HMMER's default inclusion threshold)

evalue_threshold = 0.01
hits_above_threshold = hits_df[hits_df['full_seq_evalue'] >= evalue_threshold]
hits_below_threshold = hits_df[hits_df['full_seq_evalue'] < evalue_threshold]

print(f"E-value threshold used: {evalue_threshold}")
print(f"Total sequences in alignment: {num_sequences}")
print(f"Hits with E-value < {evalue_threshold} (should be ALL sequences): {len(hits_below_threshold)}")
print(f"Hits with E-value >= {evalue_threshold} (should be ZERO):           {len(hits_above_threshold)}")
print()

if len(hits_below_threshold) == num_sequences and len(hits_above_threshold) == 0:
    print("VALIDATION PASSED: All input sequences score well against their own HMM.")
    print()
    print("Score summary:")
    print(f"  Minimum full_seq_score : {hits_df['full_seq_score'].min():.1f} bits")
    print(f"  Maximum full_seq_score : {hits_df['full_seq_score'].max():.1f} bits")
    print(f"  Minimum E-value        : {hits_df['full_seq_evalue'].min():.2e}")
    print(f"  Maximum E-value        : {hits_df['full_seq_evalue'].max():.2e}")
elif len(hits_below_threshold) == 0:
    print("WARNING: No hits found. Check that hmmbuild and hmmpress completed successfully.")
else:
    print(f"WARNING: {len(hits_above_threshold)} sequence(s) scored above the E-value threshold.")
    print("These sequences may have been poorly represented in the alignment.")
    print()
    print("Problematic sequences:")
    print(hits_above_threshold[['sequence', 'full_seq_evalue', 'full_seq_score']].to_string(index=False))

**Question:** All sequences in the alignment score very well (tiny E-values). Is this surprising? What would it mean if some globins in the alignment had E-values > 0.01?

*Answer hint:* It is expected that sequences score well against a model built from them — this is circular by design. The self-search is not a test of generalizability; it is a sanity check that the HMM was built correctly and can recognise the exact input sequences. If a sequence in the alignment scored poorly (E > 0.01), it could mean: (1) that sequence is an outlier that is too divergent from the others to be well modeled, (2) the alignment contains errors at that row, or (3) there is a problem with the hmmbuild/hmmpress run. A more meaningful test (generalizability to new sequences) requires held-out sequences or comparison against known true positives from Pfam benchmarks.

---

## Section 7: Resources

| Resource | URL / Reference |
|----------|-----------------|
| HMMER User Guide | http://hmmer.org/documentation.html |
| HMMER .hmm file format | HMMER User Guide, Appendix B |
| Pfam database | https://www.ebi.ac.uk/interpro/entry/pfam/ |
| Eddy SR (2011) HMMER3 paper | PLoS Comput Biol 7:e1002195 |
| Durbin et al. (1998) | Chapter 5, Cambridge University Press |